# Memulai dengan Agen: Solusi Latihan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/solusi/hari3/03-agen-solusi.ipynb)

Ini **solusi lengkap** semua latihan bab ini (2 latihan) di Google Colab. Semua bagian `TODO` sudah terisi, jadi notebook ini tinggal dijalankan (**Runtime > Run all**, atau `Shift+Enter` per cell). Keluaran dari satu kali eksekusi dan penjelasan tiap latihan sudah disertakan; jawaban model berubah tiap kali dijalankan, jadi hasil Anda bisa sedikit berbeda. Kerjakan dulu [notebook latihannya](https://colab.research.google.com/github/dteti-workshop/workshop-agent-migas/blob/main/colab/hari3/03-agen-latihan.ipynb); buka solusi ini untuk mencocokkan hasil Anda atau saat macet.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-agent-migas.git"
REPO = "/content/workshop-agent-migas" if os.path.isdir("/content") else os.path.abspath("workshop-agent-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

## Bab 3 - Solusi Latihan: Agen Analis Produksi
Jalankan sel satu per satu dengan Shift+Enter

Agen dengan `create_agent()` yang menjawab pertanyaan multi-langkah dengan memadukan
pencarian dokumen, status sumur, perhitungan water cut, dan konversi satuan.

In [ ]:
# Langkah 1 - Menyiapkan model dan agen
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

from migas_shared.rag_tools import KonteksPengguna, cari_dokumen
from migas_shared.tampil import jejak_tool
from migas_shared.field_tools import TOOL_LAPANGAN

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

agent = create_agent(
    model,
    tools=[*TOOL_LAPANGAN, cari_dokumen],
    system_prompt=(
        "Anda adalah analis produksi migas. Untuk angka sumur gunakan tool status sumur; untuk hitungan "
        "gunakan tool, jangan menghitung sendiri; untuk prosedur dan batas alarm gunakan cari_dokumen dan "
        "sebut ID dokumennya. Jelaskan langkah Anda secara ringkas dalam Bahasa Indonesia."
    ),
    context_schema=KonteksPengguna,
)

## Langkah 2 - Status sumur lalu perhitungan

In [ ]:
query = "Sumur SR-020 dan ML-012: berapa total produksi minyak keduanya, dan berapa water cut masing-masing?"
print(f"Pengguna: {query}")
response = agent.invoke({"messages": [HumanMessage(content=query)]}, context=KonteksPengguna(role="engineer"))
print("  Langkah  :")
for baris in jejak_tool(response["messages"]):
    print("    -", baris)
print(f"  Jawaban  : {response['messages'][-1].content}\n")

Pengguna: Sumur SR-020 dan ML-012: berapa total produksi minyak keduanya, dan berapa water cut masing-masing?
  Langkah  :
    - cek_status_sumur({'kode_sumur': 'SR-020'})
    - cek_status_sumur({'kode_sumur': 'ML-012'})
    - hitung_water_cut({'oil_bopd': 120, 'water_bwpd': 480})
    - hitung_water_cut({'oil_bopd': 85, 'water_bwpd': 340})
  Jawaban  : Berikut ringkasan hasilnya:

###  Total Produksi Minyak
- **SR-020**: 120 bopd
- **ML-012**: 85 bopd
- **Total**: **205 bopd**

###  Water Cut (Kadar Air) Masing-masing
- **SR-020**: **80,0%** (dari total cairan 600 bpd)
- **ML-012**: **80,0%** (dari total cairan 425 bpd)

Kedua sumur sama-sama memiliki water cut 80%.



## Langkah 3 - Status sumur lalu konversi satuan

In [ ]:
query = "Berapa produksi minyak SR-020 dalam meter kubik per hari?"
print(f"Pengguna: {query}")
response = agent.invoke({"messages": [HumanMessage(content=query)]}, context=KonteksPengguna(role="engineer"))
print("  Langkah  :")
for baris in jejak_tool(response["messages"]):
    print("    -", baris)
print(f"  Jawaban  : {response['messages'][-1].content}\n")

Pengguna: Berapa produksi minyak SR-020 dalam meter kubik per hari?
  Langkah  :
    - cek_status_sumur({'kode_sumur': 'SR-020'})
    - konversi_satuan({'nilai': 120, 'dari': 'bbl', 'ke': 'm3'})
  Jawaban  : Produksi minyak sumur **SR-020** adalah **120 bopd**, yang setara dengan sekitar **19,08 m³ per hari**.

**Ringkasan langkah:**
1. Mengecek status sumur SR-020 melalui tool `cek_status_sumur` -> diperoleh produksi minyak **120 bopd**.
2. Mengonversi satuan dari barel (bbl) ke meter kubik (m³) menggunakan tool `konversi_satuan` -> **120 bbl = 19,078 m³**.

Jadi, produksi minyak SR-020 dalam meter kubik per hari adalah **19,08 m³/hari**.



## Langkah 4 - Pencarian dokumen lalu ringkasan

In [ ]:
query = "Menurut SOP H2S yang berlaku, apa tindakan saat alarm tingkat 2 berbunyi di wellhead?"
print(f"Pengguna: {query}")
response = agent.invoke({"messages": [HumanMessage(content=query)]}, context=KonteksPengguna(role="engineer"))
print("  Langkah  :")
for baris in jejak_tool(response["messages"]):
    print("    -", baris)
print(f"  Jawaban  : {response['messages'][-1].content}\n")

Pengguna: Menurut SOP H2S yang berlaku, apa tindakan saat alarm tingkat 2 berbunyi di wellhead?
  index 'sop_markdown_800': 93 chunk, 3.4s
  Langkah  :
    - cari_dokumen({'query': 'alarm H2S tingkat 2 wellhead tindakan SOP'})
  Jawaban  : Saya telah menemukan dokumen SOP yang relevan. Berikut ringkasan jawabannya berdasarkan **SOP-HSE-001 (Rev. 3)** bagian "4. Tingkat Alarm dan Tindakan" dan "6. Prosedur Evakuasi":

**Tindakan saat Alarm Tingkat 2 (merah, ≥ 10 ppm) berbunyi di wellhead:**

1. **Hentikan semua pekerjaan dan evakuasi** ke muster point yang berada di arah **atas angin (upwind)** — jarak minimum **50 m**.
2. Bergerak **tegak lurus terhadap arah angin** menuju muster point upwind.
3. **Lapor** ke supervisor/control room melalui radio: lokasi, pembacaan ppm, dan jumlah orang.
4. Lakukan **head count** di muster point.
5. **Jangan kembali** ke area sebelum pembacaan **< 5 ppm** dikonfirmasi oleh petugas gas test. Masuk kembali hanya dengan **SCBA**.

Referensi dokumen: **[SO

### Penjelasan Solusi Latihan 3.1: agen analis produksi

Loop yang ditulis tangan di bab 2 kini dikerjakan `create_agent`. Anda cukup menyerahkan model, daftar tool, dan system prompt.

- `TODO(B3.1)`: `tools=[*TOOL_LAPANGAN, cari_dokumen]` menggabungkan tool lapangan dengan pencarian dokumen. `context_schema=KonteksPengguna` memberi tool akses ke data pengguna yang tidak dilihat model.
- `TODO(B3.2)`: `context=KonteksPengguna(role="engineer")` dikirim saat `invoke`. Peran ini yang dipakai `cari_dokumen` untuk menyaring dokumen mana yang boleh dibaca.

`jejak_tool` mencetak tool apa saja yang dipanggil. Pada pertanyaan pertama agen memanggil status sumur dua kali lalu water cut dua kali, dan baru menjumlahkan produksi minyak (205 bopd). Pada pertanyaan kedua ia memanggil `cek_status_sumur` lalu `konversi_satuan`; angka 19,08 m3 berasal dari konversi 120 bbl, jadi satuan "per hari" dibawa dari bopd oleh model. Pertanyaan ketiga memanggil `cari_dokumen` sekali, dan baris `index ... 93 chunk` muncul karena indeks dibangun pada pemakaian pertama.

## Bab 3 - Solusi Latihan Bonus: Agen Perencana Pekerjaan Pulling
Jalankan sel satu per satu dengan Shift+Enter

Agen dengan beberapa tool khusus yang butuh reasoning multi-langkah:
jadwal rig, perkiraan biaya, dan perkiraan produksi yang hilang.

In [ ]:
# Langkah 1 - Menyiapkan model
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from migas_shared.tampil import jejak_tool
from migas_shared.field_tools import cek_status_sumur

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Tool perencana pulling
Data fiktif untuk simulasi, lalu tiga tool: jadwal rig, biaya pulling, dan kehilangan produksi.

In [ ]:
# Data fiktif untuk simulasi
JADWAL_RIG = {"KN-033": 12, "SR-020": 5, "ML-012": 20}  # hari menunggu rig tersedia
BIAYA_PULLING_JUTA = {"SRP": 350, "ESP": 1200}  # juta rupiah per pekerjaan


@tool
def cek_jadwal_rig(kode_sumur: str) -> str:
    """Cek berapa hari lagi rig pulling tersedia untuk sebuah sumur.

    Args:
        kode_sumur: Kode sumur, contoh KN-033
    """
    if kode_sumur not in JADWAL_RIG:
        return f"Belum ada jadwal rig untuk {kode_sumur}."
    return f"Rig tersedia untuk {kode_sumur} dalam {JADWAL_RIG[kode_sumur]} hari."


@tool
def estimasi_biaya_pulling(jenis_pompa: str) -> str:
    """Perkirakan biaya satu pekerjaan pulling berdasarkan jenis pompa (SRP atau ESP), dalam juta rupiah.

    Args:
        jenis_pompa: SRP atau ESP
    """
    biaya = BIAYA_PULLING_JUTA.get(jenis_pompa.upper())
    if biaya is None:
        return f"Jenis pompa {jenis_pompa} tidak dikenal. Pilihan: {', '.join(BIAYA_PULLING_JUTA)}"
    return f"Biaya pulling {jenis_pompa.upper()}: sekitar Rp {biaya} juta"


@tool
def estimasi_kehilangan_produksi(minyak_bopd: float, hari: float) -> str:
    """Perkirakan produksi minyak yang hilang selama sumur berhenti: bopd dikali jumlah hari.

    Args:
        minyak_bopd: Produksi minyak normal dalam bopd
        hari: Lama sumur berhenti dalam hari
    """
    return f"Kehilangan produksi sekitar {minyak_bopd * hari:,.0f} bbl selama {hari:g} hari"

## Langkah 3 - Membuat agen perencana

In [ ]:
agent = create_agent(
    model,
    tools=[cek_status_sumur, cek_jadwal_rig, estimasi_biaya_pulling, estimasi_kehilangan_produksi],
    system_prompt=(
        "Anda adalah perencana pekerjaan sumur. Susun rencana langkah demi langkah dengan tool. "
        "Produksi normal sumur yang menunggu pulling dapat diambil dari catatan sebelum berhenti; "
        "bila tidak ada datanya, katakan asumsi yang Anda pakai. Jawab dalam Bahasa Indonesia."
    ),
)

## Langkah 4 - Rencana pulling satu sumur

In [ ]:
query = (
    "Rencanakan pulling sumur KN-033: kapan rig tersedia, berapa kira-kira biayanya, "
    "dan berapa produksi hilang selama menunggu jika produksi normalnya 45 bopd?"
)
print(f"Pengguna: {query}")
response = agent.invoke({"messages": [HumanMessage(content=query)]})
print("  Langkah:")
for baris in jejak_tool(response["messages"]):
    print("    -", baris)
print(f"  Rencana: {response['messages'][-1].content}\n")

Pengguna: Rencanakan pulling sumur KN-033: kapan rig tersedia, berapa kira-kira biayanya, dan berapa produksi hilang selama menunggu jika produksi normalnya 45 bopd?
  Langkah:
    - cek_status_sumur({'kode_sumur': 'KN-033'})
    - cek_jadwal_rig({'kode_sumur': 'KN-033'})
    - estimasi_biaya_pulling({'jenis_pompa': 'SRP'})
    - estimasi_kehilangan_produksi({'minyak_bopd': 45, 'hari': 12})
  Rencana: Berikut adalah rencana pulling untuk sumur **KN-033**:

---

##  Rencana Pulling Sumur KN-033

### 1. Status Sumur Saat Ini
- **Lokasi/Area**: Kenanga
- **Jenis Pompa**: SRP (Sucker Rod Pump)
- **Status**: Menunggu pulling
- **Penyebab**: Sucker rod patah
- **Produksi saat ini**: 0 bopd / 0 bwpd (berhenti berproduksi)

### 2. Jadwal Rig
- **Rig tersedia dalam**: **12 hari** dari sekarang
- Pulling dapat dilaksanakan setelah hari ke-12.

### 3. Perkiraan Biaya Pulling
- **Jenis pekerjaan**: Pulling SRP
- **Estimasi biaya**: **± Rp 350 juta**
  *(Biaya ini mencakup mobilisasi rig, operasi p

## Langkah 5 - Membandingkan dua sumur

In [ ]:
query = "Bandingkan KN-033 dan SR-020: sumur mana yang rig-nya lebih cepat tersedia dan berapa biaya pulling masing-masing?"
print(f"Pengguna: {query}")
response = agent.invoke({"messages": [HumanMessage(content=query)]})
print("  Langkah:")
for baris in jejak_tool(response["messages"]):
    print("    -", baris)
print(f"  Rencana: {response['messages'][-1].content}\n")

Pengguna: Bandingkan KN-033 dan SR-020: sumur mana yang rig-nya lebih cepat tersedia dan berapa biaya pulling masing-masing?
  Langkah:
    - cek_status_sumur({'kode_sumur': 'KN-033'})
    - cek_status_sumur({'kode_sumur': 'SR-020'})
    - cek_jadwal_rig({'kode_sumur': 'KN-033'})
    - cek_jadwal_rig({'kode_sumur': 'SR-020'})
    - estimasi_biaya_pulling({'jenis_pompa': 'SRP'})
    - estimasi_biaya_pulling({'jenis_pompa': 'ESP'})
  Rencana: Berikut perbandingan lengkapnya:

---

##  Perbandingan KN-033 vs SR-020

| Parameter | **KN-033 (Kenanga)** | **SR-020 (Seruni)** |
|:---|:---:|:---:|
| **Jenis Pompa** | SRP (Sucker Rod Pump) | ESP (Electric Submersible Pump) |
| **Status Saat Ini** |  Menunggu pulling |  Masih produksi (stabil) |
| **Masalah** | Sucker rod patah | Trip overload kemarin, sudah stabil |
| **Rig Tersedia dalam** | **12 hari** | **5 hari** |
| **Biaya Pulling** | **Rp 350 juta** | **Rp 1.200 juta** |

---

### Kesimpulan:

1. **Rig lebih cepat tersedia untuk SR-020 (

### Penjelasan Solusi Latihan 3.2: agen perencana pulling

Kali ini yang Anda tulis adalah tool-nya, dan agen yang merangkainya. Docstring tool berfungsi sebagai petunjuk pemakaian bagi model.

- `TODO(B3.3)` dan `TODO(B3.4)`: dua tool sederhana. `cek_jadwal_rig` mengembalikan pesan jelas bila sumur tidak ada di `JADWAL_RIG`; pesan itu yang akan dibaca model, bukan error Python.
- `TODO(B3.5)`: agen menerima empat tool (tiga buatan sendiri dan `cek_status_sumur`) dan system prompt yang meminta rencana langkah demi langkah serta asumsi yang disebutkan bila data kurang.

Pada keluaran, rencana KN-033 memanggil empat tool: status sumur untuk mengetahui jenis pompanya (SRP), jadwal rig (12 hari), biaya pulling SRP (sekitar Rp 350 juta), dan kehilangan produksi (45 bopd kali 12 hari). Perbandingan dua sumur memanggil enam tool dan menyimpulkan SR-020 lebih cepat (5 hari dibanding 12) tetapi lebih mahal (ESP, Rp 1.200 juta). Pemilihan `SRP` atau `ESP` untuk `jenis_pompa` dilakukan model dari hasil status sumur; itu alasan status sumur dipanggil lebih dulu.